# 86BB Cluster A/V Profiling

Builds on the gut state mechanics (notebook 03) to:
1. Load untapped blood marker data from PREDICT 1 supplementary tables
2. Expand diet archetypes and improve PCA space coverage
3. Profile each cluster with metabolic, neuroactive, and dietary evidence
4. Export comprehensive cluster profiles for A/V preset design

In [1]:
import zipfile
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

PLOTLY_DARK = 'plotly_dark'
OUT_DIR = Path('../outputs')
SUPP_DIR = Path('../data/predict1_paper/supplementary')
TABLE5_FILE = SUPP_DIR / '41591_2020_1183_MOESM6_ESM.xlsx'
TABLE9_FILE = SUPP_DIR / '41591_2020_1183_MOESM10_ESM.xlsx'

important_species = pd.read_csv(OUT_DIR / 'important_species_list.csv')['species'].tolist()
corr_matrix = pd.read_csv(OUT_DIR / 'predict1_food_species_correlations.csv', index_col=0)
health_dir_df = pd.read_csv(OUT_DIR / 'health_direction_vector.csv', index_col=0)
food_profiles_df = pd.read_csv(OUT_DIR / 'food_species_profiles.csv', index_col=0)
weight_matrix = pd.read_csv(OUT_DIR / 'food_category_weights.csv', index_col=0)

FOOD_GROUPS = list(corr_matrix.columns)
corr_np = corr_matrix.loc[important_species].values.T

t9 = pd.read_excel(TABLE9_FILE, sheet_name='Ranks', skiprows=3, engine='openpyxl')
t9.columns = ['species'] + list(t9.columns[1:])
t9 = t9[t9['species'].notna() & (t9['species'] != 'Category')].reset_index(drop=True)
t9['Final Rank'] = pd.to_numeric(t9['Final Rank'], errors='coerce')
t9 = t9.dropna(subset=['Final Rank']).set_index('species')

health_direction = health_dir_df['health_weight'].reindex(important_species).fillna(0)

print(f'Loaded: {len(important_species)} important species, {len(FOOD_GROUPS)} food groups')
print(f'Table 9: {len(t9)} species with {len(t9.columns)} columns')
print(f'Table 9 columns: {list(t9.columns)}')

Loaded: 135 important species, 20 food groups
Table 9: 176 species with 24 columns
Table 9 columns: ['quicki_score', 'amed_score', 'HFD', 'hei_score', 'HDL_size_0', 'PUFA_pct_0', 'HDL_size_360', 'ASCVD_10yr_risk', 'visceral_fat', 'LiverFatProbability', 'uPDI', 'Total_TG_0', 'VLDL_size_0', 'GlycA_0', 'Meal_JJ_Hospital__meal_glucose_120_iauc', 'Meal_JJ_Hospital__meal_c-peptide_120_iauc', 'Meal_JJ_Hospital__meal_trig_360_iauc', 'GlycA_360', 'VLDL_size_360', 'Fasting', 'Habitual Diet', 'Personal', 'Postprandial', 'Final Rank']


---
## 1. Load Blood Marker Data (Table 5, Sheets 3 & 4)

In [2]:
NS = {'main': 'http://purl.oclc.org/ooxml/spreadsheetml/main'}

def load_strict_xlsx(filepath, sheet_index=0):
    with zipfile.ZipFile(filepath) as z:
        ss_xml = z.read('xl/sharedStrings.xml').decode('utf-8')
        ss_root = ET.fromstring(ss_xml)
        strings = []
        for si in ss_root.findall('.//main:si', NS):
            parts = [t.text for t in si.findall('.//main:t', NS) if t.text]
            strings.append(''.join(parts))

        sheet_file = f'xl/worksheets/sheet{sheet_index + 1}.xml'
        xml = z.read(sheet_file).decode('utf-8')
        root = ET.fromstring(xml)

        rows = []
        for row_el in root.findall('.//main:row', NS):
            cells = {}
            for cell in row_el.findall('main:c', NS):
                ref = cell.attrib.get('r', '')
                col = ''.join(c for c in ref if c.isalpha())
                val_el = cell.find('main:v', NS)
                if val_el is None:
                    continue
                cell_type = cell.attrib.get('t', '')
                if cell_type == 's':
                    cells[col] = strings[int(val_el.text)]
                else:
                    try:
                        cells[col] = float(val_el.text)
                    except ValueError:
                        cells[col] = val_el.text
            if cells:
                rows.append(cells)
        return rows

def rows_to_df(rows, skip_header=3):
    data = rows[skip_header:]
    df = pd.DataFrame(data)
    df.columns = ['variable', 'species', 'spearman_r', 'pvalue', 'qvalue']
    df['spearman_r'] = pd.to_numeric(df['spearman_r'], errors='coerce')
    df['pvalue'] = pd.to_numeric(df['pvalue'], errors='coerce')
    df['qvalue'] = pd.to_numeric(df['qvalue'], errors='coerce')
    return df.dropna(subset=['spearman_r'])

fasting_df = rows_to_df(load_strict_xlsx(TABLE5_FILE, 3))
postprandial_df = rows_to_df(load_strict_xlsx(TABLE5_FILE, 4))

fasting_matrix = fasting_df.pivot(index='species', columns='variable', values='spearman_r')
fasting_q = fasting_df.pivot(index='species', columns='variable', values='qvalue')
postprandial_matrix = postprandial_df.pivot(index='species', columns='variable', values='spearman_r')
postprandial_q = postprandial_df.pivot(index='species', columns='variable', values='qvalue')

print(f'Fasting markers: {fasting_matrix.shape} (species x markers)')
print(f'Postprandial markers: {postprandial_matrix.shape} (species x markers)')
print(f'\nFasting marker names ({len(fasting_matrix.columns)}):')
for i, c in enumerate(sorted(fasting_matrix.columns)):
    print(f'  {c}')

Fasting markers: (176, 75) (species x markers)
Postprandial markers: (176, 169) (species x markers)

Fasting marker names (75):
  ApoA1_0
  ApoB_0
  ApoB_by_ApoA1_0
  CPEP_0
  Clinical_LDL_C_0
  GlycA_0
  HDL_C_0
  HDL_L_0
  HDL_TG_0
  HDL_size_0
  IDL_C_0
  IDL_L_0
  IDL_TG_0
  IL-6_0
  INS_0
  LDL_C_0
  LDL_L_0
  LDL_TG_0
  LDL_size_0
  L_HDL_C_0
  L_HDL_L_0
  L_HDL_TG_0
  L_LDL_C_0
  L_LDL_L_0
  L_LDL_TG_0
  L_VLDL_C_0
  L_VLDL_L_0
  L_VLDL_TG_0
  MUFA_pct_0
  M_HDL_C_0
  M_HDL_L_0
  M_HDL_TG_0
  M_LDL_C_0
  M_LDL_L_0
  M_LDL_TG_0
  M_VLDL_C_0
  M_VLDL_L_0
  M_VLDL_TG_0
  Omega_3_pct_0
  Omega_6_pct_0
  PGLU_0
  PUFA_pct_0
  Remnant_C_0
  SFA_pct_0
  S_HDL_C_0
  S_HDL_L_0
  S_HDL_TG_0
  S_HDL_TG_pct_0
  S_LDL_C_0
  S_LDL_L_0
  S_LDL_TG_0
  S_VLDL_C_0
  S_VLDL_L_0
  S_VLDL_TG_0
  Total_C_0
  Total_FA_0
  Total_L_0
  Total_TG_0
  VLDL_C_0
  VLDL_L_0
  VLDL_TG_0
  VLDL_size_0
  XL_HDL_C_0
  XL_HDL_L_0
  XL_HDL_TG_0
  XL_VLDL_C_0
  XL_VLDL_L_0
  XL_VLDL_TG_0
  XS_VLDL_C_0
  XS_VLDL_L_0


---
## 2. Table 9 Expanded Health Profile

In [3]:
numeric_cols = []
for c in t9.columns:
    t9[c] = pd.to_numeric(t9[c], errors='coerce')
    if t9[c].notna().sum() > 100:
        numeric_cols.append(c)

print(f'Table 9 numeric columns ({len(numeric_cols)}):')
for c in numeric_cols:
    print(f'  {c}: {t9[c].notna().sum()} values, range [{t9[c].min():.1f}, {t9[c].max():.1f}]')

Table 9 numeric columns (24):
  quicki_score: 176 values, range [1.0, 176.0]
  amed_score: 176 values, range [1.0, 176.0]
  HFD: 176 values, range [1.0, 176.0]
  hei_score: 176 values, range [1.0, 176.0]
  HDL_size_0: 176 values, range [1.0, 176.0]
  PUFA_pct_0: 176 values, range [1.0, 176.0]
  HDL_size_360: 176 values, range [1.0, 176.0]
  ASCVD_10yr_risk: 176 values, range [1.0, 176.0]
  visceral_fat: 176 values, range [1.0, 176.0]
  LiverFatProbability: 176 values, range [1.0, 176.0]
  uPDI: 176 values, range [1.0, 176.0]
  Total_TG_0: 176 values, range [1.0, 176.0]
  VLDL_size_0: 176 values, range [1.0, 176.0]
  GlycA_0: 176 values, range [1.0, 176.0]
  Meal_JJ_Hospital__meal_glucose_120_iauc: 176 values, range [1.0, 176.0]
  Meal_JJ_Hospital__meal_c-peptide_120_iauc: 176 values, range [1.0, 176.0]
  Meal_JJ_Hospital__meal_trig_360_iauc: 176 values, range [1.0, 176.0]
  GlycA_360: 176 values, range [1.0, 176.0]
  VLDL_size_360: 176 values, range [1.0, 176.0]
  Fasting: 176 values, 

---
## 3. Neuroactive Pathway Mapping (Literature-Based)

Species-to-pathway assignments based on published gut-brain axis research (Valles-Colomer et al. 2019, Strandwitz 2018, Yano et al. 2015, and others).

In [4]:
NEUROACTIVE_PATHWAYS = {
    'butyrate_SCFA': [
        'Faecalibacterium_prausnitzii', 'Roseburia_hominis', 'Roseburia_intestinalis',
        'Roseburia_faecis', 'Roseburia_sp_CAG_182', 'Roseburia_sp_CAG_309',
        'Roseburia_sp_CAG_471', 'Eubacterium_hallii', 'Eubacterium_eligens',
        'Coprococcus_eutactus', 'Coprococcus_comes', 'Coprococcus_catus',
        'Anaerostipes_hadrus', 'Agathobaculum_butyriciproducens',
        'Lachnospira_pectinoschiza', 'Eubacterium_ramulus',
        'Intestinimonas_butyriciproducens',
    ],
    'GABA': [
        'Bifidobacterium_longum', 'Bifidobacterium_adolescentis',
        'Bifidobacterium_bifidum', 'Bifidobacterium_catenulatum',
        'Bifidobacterium_pseudocatenulatum', 'Bifidobacterium_animalis',
        'Lactococcus_lactis', 'Bacteroides_fragilis',
    ],
    'tryptophan_serotonin': [
        'Streptococcus_thermophilus', 'Streptococcus_salivarius',
        'Streptococcus_parasanguinis', 'Eubacterium_hallii',
        'Faecalibacterium_prausnitzii', 'Roseburia_hominis',
        'Coprococcus_eutactus', 'Coprococcus_comes',
    ],
    'dopamine_catecholamine': [
        'Escherichia_coli',
    ],
    'pro_inflammatory': [
        'Bilophila_wadsworthia', 'Ruminococcus_gnavus', 'Ruminococcus_torques',
        'Clostridium_bolteae', 'Clostridium_bolteae_CAG_59', 'Clostridium_symbiosum',
        'Hungatella_hathewayi', 'Escherichia_coli', 'Clostridium_innocuum',
        'Flavonifractor_plautii', 'Eggerthella_lenta',
    ],
    'gut_barrier': [
        'Akkermansia_muciniphila', 'Faecalibacterium_prausnitzii',
        'Bifidobacterium_longum', 'Bifidobacterium_adolescentis',
        'Bifidobacterium_bifidum', 'Roseburia_intestinalis',
    ],
}

for pathway, species_list in NEUROACTIVE_PATHWAYS.items():
    present = [s for s in species_list if s in important_species]
    missing = [s for s in species_list if s not in important_species]
    NEUROACTIVE_PATHWAYS[pathway] = present
    print(f'{pathway}: {len(present)}/{len(present)+len(missing)} species in our 135')
    if missing:
        print(f'  Missing: {missing}')

butyrate_SCFA: 17/17 species in our 135
GABA: 8/8 species in our 135
tryptophan_serotonin: 8/8 species in our 135
dopamine_catecholamine: 1/1 species in our 135
pro_inflammatory: 11/11 species in our 135
gut_barrier: 6/6 species in our 135


---
## 4. PCA Loading Analysis

In [5]:
def make_diet(weights_dict):
    v = np.zeros(len(FOOD_GROUPS))
    for g, w in weights_dict.items():
        v[FOOD_GROUPS.index(g)] = w
    return v / v.sum()

ARCHETYPES_ORIGINAL = {
    'Mediterranean':     make_diet({'Vegetables': 3, 'Fruits': 2, 'Fish_seafood': 2, 'Nuts': 2, 'Whole_grain': 2, 'Vegetable_oils': 2, 'Legumes': 1.5, 'Alcohol': 1}),
    'Western processed': make_diet({'Refined_grains': 3, 'Meat': 3, 'Sweets_desserts': 2.5, 'Sugar_sweetened_beverages': 2, 'Animal_fats': 2, 'Potatoes': 1.5, 'Dairy': 1}),
    'Plant-based':       make_diet({'Vegetables': 4, 'Fruits': 3, 'Legumes': 3, 'Whole_grain': 3, 'Nuts': 2, 'Vegetable_oils': 1}),
    'Balanced':          make_diet({g: 1 for g in FOOD_GROUPS}),
    'Carnivore':         make_diet({'Meat': 5, 'Animal_fats': 3, 'Egg': 2, 'Fish_seafood': 1, 'Dairy': 1}),
    'Junk food':         make_diet({'Sweets_desserts': 3, 'Refined_grains': 3, 'Sugar_sweetened_beverages': 3, 'Potatoes': 2, 'Animal_fats': 2, 'Meat': 1}),
    'Japanese':          make_diet({'Fish_seafood': 3, 'Vegetables': 2.5, 'Legumes': 2, 'Refined_grains': 2, 'Whole_grain': 1, 'Tea_coffee': 1.5, 'Egg': 0.5}),
    'Korean':            make_diet({'Vegetables': 3, 'Legumes': 2, 'Refined_grains': 2, 'Fish_seafood': 1.5, 'Meat': 1, 'Tea_coffee': 1}),
    'South Indian':      make_diet({'Legumes': 3, 'Vegetables': 3, 'Whole_grain': 2, 'Vegetable_oils': 2, 'Fruits': 1, 'Tea_coffee': 1}),
    'Chinese urban':     make_diet({'Vegetables': 2.5, 'Refined_grains': 2.5, 'Meat': 2, 'Vegetable_oils': 2, 'Legumes': 1, 'Fish_seafood': 1, 'Tea_coffee': 1}),
    'Thai':              make_diet({'Vegetables': 3, 'Fish_seafood': 2, 'Refined_grains': 2, 'Legumes': 1.5, 'Fruits': 1, 'Vegetable_oils': 1, 'Nuts': 0.5}),
    'Italian':           make_diet({'Vegetables': 2.5, 'Refined_grains': 2.5, 'Vegetable_oils': 2.5, 'Fruits': 1.5, 'Dairy': 1, 'Meat': 1, 'Alcohol': 1}),
    'French':            make_diet({'Dairy': 2.5, 'Vegetables': 2, 'Meat': 2, 'Refined_grains': 2, 'Alcohol': 2, 'Animal_fats': 1.5, 'Fruits': 1}),
    'Nordic':            make_diet({'Fish_seafood': 3, 'Whole_grain': 3, 'Vegetables': 2, 'Dairy': 2, 'Fruits': 1, 'Nuts': 1}),
    'Mexican':           make_diet({'Legumes': 3, 'Vegetables': 2, 'Refined_grains': 2, 'Meat': 1.5, 'Fruits': 1.5, 'Vegetable_oils': 1}),
    'Brazilian':         make_diet({'Legumes': 2.5, 'Meat': 2, 'Refined_grains': 2, 'Vegetables': 2, 'Fruits': 2, 'Vegetable_oils': 1}),
    'US fast food':      make_diet({'Refined_grains': 3, 'Meat': 3, 'Potatoes': 2.5, 'Sugar_sweetened_beverages': 2.5, 'Animal_fats': 2, 'Sweets_desserts': 1}),
    'British pub':       make_diet({'Meat': 3, 'Potatoes': 3, 'Animal_fats': 2, 'Alcohol': 2.5, 'Refined_grains': 2, 'Dairy': 1}),
    'Keto':              make_diet({'Animal_fats': 4, 'Meat': 3, 'Egg': 2, 'Vegetables': 1.5, 'Nuts': 1.5, 'Dairy': 1}),
    'Raw vegan':         make_diet({'Vegetables': 4, 'Fruits': 4, 'Nuts': 3, 'Legumes': 1}),
    'Dairy-heavy':       make_diet({'Dairy': 5, 'Fruits': 1, 'Whole_grain': 1, 'Vegetables': 1}),
    'High fiber':        make_diet({'Whole_grain': 4, 'Legumes': 3, 'Vegetables': 3, 'Fruits': 2, 'Nuts': 1}),
    'Sugar addict':      make_diet({'Sugar_sweetened_beverages': 4, 'Sweets_desserts': 4, 'Fruit_juices': 2, 'Refined_grains': 2}),
    'Pescatarian':       make_diet({'Fish_seafood': 3, 'Vegetables': 3, 'Whole_grain': 2, 'Fruits': 2, 'Legumes': 1, 'Nuts': 1}),
    'Middle Eastern':    make_diet({'Legumes': 3, 'Vegetables': 2.5, 'Whole_grain': 2, 'Vegetable_oils': 2, 'Dairy': 1, 'Nuts': 1, 'Fruits': 1}),
    'Ethiopian':         make_diet({'Legumes': 3, 'Whole_grain': 3, 'Vegetables': 2.5, 'Vegetable_oils': 1.5, 'Meat': 0.5}),
    'West African':      make_diet({'Legumes': 2.5, 'Whole_grain': 2, 'Vegetables': 2, 'Fish_seafood': 1.5, 'Vegetable_oils': 1.5, 'Fruits': 1.5, 'Potatoes': 1}),
}

def build_profiles(archetypes, n_variants=40, n_blends=500):
    profiles, labels, types, diet_vecs = [], [], [], []
    names = list(archetypes.keys())
    vecs = np.array(list(archetypes.values()))
    for name, dv in archetypes.items():
        for _ in range(n_variants):
            noisy = dv + np.random.normal(0, 0.02, len(dv))
            noisy = np.clip(noisy, 0, None); noisy /= noisy.sum()
            profiles.append(noisy @ corr_np)
            labels.append(name); types.append('archetype'); diet_vecs.append(noisy)
    for _ in range(n_blends):
        i1, i2 = np.random.choice(len(names), 2, replace=False)
        t = np.random.beta(2, 2)
        bl = t * vecs[i1] + (1-t) * vecs[i2] + np.random.normal(0, 0.01, len(vecs[0]))
        bl = np.clip(bl, 0, None); bl /= bl.sum()
        profiles.append(bl @ corr_np)
        labels.append(f'{names[i1]}+{names[i2]}'); types.append('blend'); diet_vecs.append(bl)
    return np.array(profiles), labels, types, np.array(diet_vecs)

profiles_orig, labels_orig, types_orig, dvecs_orig = build_profiles(ARCHETYPES_ORIGINAL)

scaler_orig = StandardScaler()
scaled_orig = scaler_orig.fit_transform(profiles_orig)
pca_orig = PCA(n_components=3)
coords_orig = pca_orig.fit_transform(scaled_orig)

loadings = pd.DataFrame(pca_orig.components_.T, index=important_species, columns=['PC1','PC2','PC3'])

print(f'PCA explained variance: {pca_orig.explained_variance_ratio_}')
print(f'Cumulative: {np.cumsum(pca_orig.explained_variance_ratio_)}')
print(f'\nOriginal archetypes: {len(ARCHETYPES_ORIGINAL)}, total profiles: {len(profiles_orig)}')

PCA explained variance: [0.67583232 0.13327667 0.0701237 ]
Cumulative: [0.67583232 0.809109   0.87923269]

Original archetypes: 27, total profiles: 1580


In [6]:
var_ratios = pca_orig.explained_variance_ratio_
for i, pc in enumerate(['PC1', 'PC2', 'PC3']):
    top = loadings[pc].abs().nlargest(10)
    print(f'\n=== {pc} (explains {var_ratios[i]*100:.1f}%) ===')
    for sp in top.index:
        sign = '+' if loadings.loc[sp, pc] > 0 else '-'
        food_corrs = corr_matrix.loc[sp]
        top_food = food_corrs.abs().nlargest(3)
        foods_str = ', '.join([f'{f}({food_corrs[f]:+.3f})' for f in top_food.index])
        print(f'  {sign} {sp}: loading={loadings.loc[sp,pc]:.4f}  top foods: {foods_str}')


=== PC1 (explains 67.6%) ===
  + Parabacteroides_merdae: loading=0.1041  top foods: Vegetables(-0.098), Sweets_desserts(+0.091), Meat(+0.089)
  + Escherichia_coli: loading=0.1041  top foods: Nuts(-0.153), Vegetables(-0.087), Vegetable_oils(-0.071)
  - Roseburia_hominis: loading=-0.1036  top foods: Nuts(+0.197), Whole_grain(+0.114), Vegetable_oils(+0.092)
  + Bilophila_wadsworthia: loading=0.1036  top foods: Nuts(-0.119), Meat(+0.103), Sweets_desserts(+0.088)
  + Pseudoflavonifractor_sp_An184: loading=0.1035  top foods: Nuts(-0.137), Fruits(-0.127), Vegetables(-0.099)
  + Ruthenibacterium_lactatiformans: loading=0.1035  top foods: Nuts(-0.181), Potatoes(+0.135), Meat(+0.133)
  + Harryflintia_acetispora: loading=0.1033  top foods: Meat(+0.124), Legumes(-0.100), Nuts(-0.082)
  + Flavonifractor_plautii: loading=0.1031  top foods: Nuts(-0.120), Sugar_sweetened_beverages(+0.104), Potatoes(+0.100)
  + Eisenbergiella_tayi: loading=0.1030  top foods: Nuts(-0.088), Meat(+0.087), Vegetables(-0.0

In [7]:
km_orig = KMeans(n_clusters=12, n_init=10, random_state=42)
clabels_orig = km_orig.fit_predict(scaled_orig)
centroids_3d_orig = pca_orig.transform(km_orig.cluster_centers_)

print('Original 12 cluster centroids in PCA space:')
print(f'{"Cluster":>8} {"PC1":>8} {"PC2":>8} {"PC3":>8} {"Members":>8}')
for ci in range(12):
    n = (clabels_orig == ci).sum()
    print(f'{ci:>8} {centroids_3d_orig[ci,0]:>8.3f} {centroids_3d_orig[ci,1]:>8.3f} {centroids_3d_orig[ci,2]:>8.3f} {n:>8}')

print(f'\nPC3 range of centroids: [{centroids_3d_orig[:,2].min():.3f}, {centroids_3d_orig[:,2].max():.3f}]')
print(f'PC2 range of centroids: [{centroids_3d_orig[:,1].min():.3f}, {centroids_3d_orig[:,1].max():.3f}]')

Original 12 cluster centroids in PCA space:
 Cluster      PC1      PC2      PC3  Members
       0   -2.338    1.640   -2.106      177
       1   -8.106   -1.570    0.651      192
       2    5.319    1.077   -0.558      101
       3   -3.213    5.371   11.638       67
       4    2.170   -4.373    0.579      178
       5   14.995    9.572   -0.702       60
       6    8.549   -8.525    2.811      105
       7   -9.959    2.457   -0.994      277
       8  -14.631   -1.590   -0.559       55
       9   -2.879   -1.181   -2.011      154
      10   14.609   -4.539   -1.408       64
      11   16.911    3.472   -0.839      150

PC3 range of centroids: [-2.106, 11.638]
PC2 range of centroids: [-8.525, 9.572]


---
## 5. Expanded Archetypes

Based on PCA loading analysis, add new archetypes to fill underrepresented regions of PC2/PC3 space.

In [8]:
NEW_ARCHETYPES = {
    'Alcohol-heavy':       make_diet({'Alcohol': 5, 'Nuts': 1, 'Meat': 1}),
    'Fermented dairy':     make_diet({'Dairy': 5, 'Vegetables': 1}),
    'Tea/coffee dominant': make_diet({'Tea_coffee': 5, 'Whole_grain': 1, 'Fruits': 1}),
    'Egg-heavy':           make_diet({'Egg': 5, 'Vegetables': 1, 'Whole_grain': 1}),
    'Margarine processed': make_diet({'Margarine': 4, 'Refined_grains': 3, 'Potatoes': 2}),
    'Fruit juice heavy':   make_diet({'Fruit_juices': 4, 'Fruits': 2, 'Sweets_desserts': 1}),
    'Organ meats':         make_diet({'Misc_animal_foods': 4, 'Meat': 2, 'Animal_fats': 1}),
    'Ultra-processed':     make_diet({'Refined_grains': 4, 'Sugar_sweetened_beverages': 3, 'Sweets_desserts': 3, 'Margarine': 2, 'Potatoes': 2}),
    'Raw vegan extreme':   make_diet({'Vegetables': 5, 'Fruits': 4, 'Nuts': 3}),
    'Zero-carb':           make_diet({'Meat': 4, 'Fish_seafood': 3, 'Animal_fats': 3, 'Egg': 2}),
    'Potato-starch heavy': make_diet({'Potatoes': 5, 'Vegetables': 1, 'Animal_fats': 1}),
    'Nut-seed dominant':   make_diet({'Nuts': 5, 'Fruits': 2, 'Vegetables': 1}),
}

ALL_ARCHETYPES = {**ARCHETYPES_ORIGINAL, **NEW_ARCHETYPES}
print(f'Total archetypes: {len(ALL_ARCHETYPES)} ({len(ARCHETYPES_ORIGINAL)} original + {len(NEW_ARCHETYPES)} new)')

Total archetypes: 39 (27 original + 12 new)


---
## 6. Regenerate Profiles, Re-fit PCA, and Cluster

Build a new synthetic profile cloud from ALL archetypes, re-fit PCA, and find optimal K.

In [9]:
profiles_all, labels_all, types_all, dvecs_all = build_profiles(ALL_ARCHETYPES, n_variants=40, n_blends=700)

scaler = StandardScaler()
profiles_scaled = scaler.fit_transform(profiles_all)
pca3 = PCA(n_components=3)
coords_3d = pca3.fit_transform(profiles_scaled)

print(f'New PCA explained variance: {pca3.explained_variance_ratio_}')
print(f'Cumulative: {np.cumsum(pca3.explained_variance_ratio_)}')
print(f'Total profiles: {len(profiles_all)} ({len(ALL_ARCHETYPES)} archetypes x 40 + 700 blends)')

base_labels = [l.split('+')[0] if '+' in l else l for l in labels_all]

New PCA explained variance: [0.57818563 0.15740458 0.06461642]
Cumulative: [0.57818563 0.73559021 0.80020663]
Total profiles: 2260 (39 archetypes x 40 + 700 blends)


In [10]:
sil_scores = {}
for k in range(8, 22):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labs = km.fit_predict(profiles_scaled)
    sil = silhouette_score(profiles_scaled, labs)
    min_size = min(np.bincount(labs))
    sil_scores[k] = (sil, min_size)
    print(f'K={k:2d}: silhouette={sil:.4f}, min cluster size={min_size}')

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(sil_scores.keys()), y=[v[0] for v in sil_scores.values()],
    mode='lines+markers', name='Silhouette'))
fig.update_layout(title='Silhouette Score vs K', xaxis_title='K', yaxis_title='Silhouette Score',
    template=PLOTLY_DARK, width=700, height=400)
fig.show()

K= 8: silhouette=0.3341, min cluster size=56
K= 9: silhouette=0.3623, min cluster size=55
K=10: silhouette=0.3256, min cluster size=54
K=11: silhouette=0.3597, min cluster size=54
K=12: silhouette=0.3534, min cluster size=53
K=13: silhouette=0.3697, min cluster size=53
K=14: silhouette=0.3992, min cluster size=53
K=15: silhouette=0.3947, min cluster size=53
K=16: silhouette=0.3920, min cluster size=53
K=17: silhouette=0.3929, min cluster size=53
K=18: silhouette=0.4081, min cluster size=53
K=19: silhouette=0.4141, min cluster size=49
K=20: silhouette=0.4264, min cluster size=50
K=21: silhouette=0.4292, min cluster size=49


In [11]:
K = 16
km = KMeans(n_clusters=K, n_init=10, random_state=42)
cluster_labels = km.fit_predict(profiles_scaled)
centroids_scaled = km.cluster_centers_
centroids_3d = pca3.transform(centroids_scaled)
centroids_original = scaler.inverse_transform(centroids_scaled)

def compute_health_simple(gut_state, health_dir=health_direction.values):
    return float(gut_state @ health_dir)

health_scores_all = np.array([compute_health_simple(p) for p in profiles_all])
centroid_health = []
for ci in range(K):
    mask = cluster_labels == ci
    centroid_health.append(health_scores_all[mask].mean())

print(f'Clusters (K={K}):')
print(f'{"Cluster":>8} {"Members":>8} {"Health":>8} {"PC1":>8} {"PC2":>8} {"PC3":>8} {"Top diets"}')
for ci in range(K):
    n = (cluster_labels == ci).sum()
    top = pd.Series([base_labels[j] for j in range(len(base_labels)) if cluster_labels[j]==ci]).value_counts().head(3)
    diets_str = ', '.join([f'{d}({c})' for d,c in top.items()])
    print(f'{ci:>8} {n:>8} {centroid_health[ci]:>+8.3f} {centroids_3d[ci,0]:>8.3f} {centroids_3d[ci,1]:>8.3f} {centroids_3d[ci,2]:>8.3f}  {diets_str}')

Clusters (K=16):
 Cluster  Members   Health      PC1      PC2      PC3 Top diets
       0      103   +1.616   -2.573    4.683   10.088  Fermented dairy(49), Dairy-heavy(45), Sugar addict(2)
       1      126   +2.969  -13.175    0.161   -1.433  Raw vegan(54), Raw vegan extreme(47), Plant-based(6)
       2       55   -1.392   14.295   -0.161    2.948  Organ meats(48), Korean(2), Thai(1)
       3      176   -1.199   13.104    4.730   -0.323  Ultra-processed(57), Sugar addict(48), Junk food(45)
       4      151   +0.046    6.365   -7.326    2.250  Carnivore(47), Zero-carb(47), Keto(41)
       5      486   +2.144   -8.233    2.155   -0.618  High fiber(57), South Indian(55), Middle Eastern(55)
       6      334   +1.230   -2.495    0.818   -1.237  Korean(49), Mexican(48), Thai(47)
       7       69   +0.405    1.651    4.498   -1.873  Fruit juice heavy(55), Sugar addict(4), Plant-based(2)
       8       53   +0.622    1.783  -15.774   -4.090  Alcohol-heavy(46), Middle Eastern(1), Ultra-pro

In [12]:
plot_df = pd.DataFrame({
    'PC1': coords_3d[:,0], 'PC2': coords_3d[:,1], 'PC3': coords_3d[:,2],
    'label': labels_all, 'type': types_all, 'base_diet': base_labels,
    'health_score': health_scores_all, 'cluster': [str(c) for c in cluster_labels],
})

fig = px.scatter_3d(plot_df, x='PC1', y='PC2', z='PC3', color='cluster',
    hover_name='label', opacity=0.5, title=f'K-Means Clusters (K={K}) — Expanded Archetypes',
    template=PLOTLY_DARK, width=950, height=750,
    color_discrete_sequence=px.colors.qualitative.Set3)
fig.update_traces(marker=dict(size=3))
for ci in range(K):
    fig.add_trace(go.Scatter3d(
        x=[centroids_3d[ci,0]], y=[centroids_3d[ci,1]], z=[centroids_3d[ci,2]],
        mode='markers+text', text=[f'C{ci}'], textposition='top center',
        marker=dict(size=12, color='white', symbol='diamond',
                    line=dict(width=2, color=px.colors.qualitative.Set3[ci % len(px.colors.qualitative.Set3)])),
        showlegend=False, hovertext=f'Centroid {ci}: health={centroid_health[ci]:.3f}'))
fig.show()

---
## 7. Metabolic Fingerprint (Blood Markers)

In [13]:
fasting_135 = fasting_matrix.reindex(important_species).fillna(0)
postprandial_135 = postprandial_matrix.reindex(important_species).fillna(0)

MARKER_CATEGORIES = {
    'inflammation': [c for c in fasting_matrix.columns if any(k in c.lower() for k in ['crp', 'glyca', 'glycoprotein'])],
    'glycemic': [c for c in fasting_matrix.columns if any(k in c.lower() for k in ['glucose', 'insulin', 'hba1c', 'homa', 'c-peptide', 'c_peptide'])],
    'lipid_favorable': [c for c in fasting_matrix.columns if any(k in c.lower() for k in ['hdl'])],
    'lipid_adverse': [c for c in fasting_matrix.columns if any(k in c.lower() for k in ['ldl', 'vldl', 'triglyceride'])],
    'body_comp': [c for c in fasting_matrix.columns if any(k in c.lower() for k in ['bmi', 'visceral', 'liver', 'waist', 'fat'])],
}

pp_categories = {
    'pp_glucose': [c for c in postprandial_matrix.columns if any(k in c.lower() for k in ['glucose', 'insulin'])],
    'pp_lipid': [c for c in postprandial_matrix.columns if any(k in c.lower() for k in ['triglyceride', 'cholesterol'])],
    'pp_inflammation': [c for c in postprandial_matrix.columns if any(k in c.lower() for k in ['crp', 'glyca', 'glycoprotein', 'il-'])],
}

print('Fasting marker categories:')
for cat, markers in MARKER_CATEGORIES.items():
    print(f'  {cat}: {len(markers)} markers')
    if len(markers) == 0:
        print(f'    (no matches — checking column names...)')

print('\nPostprandial marker categories:')
for cat, markers in pp_categories.items():
    print(f'  {cat}: {len(markers)} markers')

if sum(len(v) for v in MARKER_CATEGORIES.values()) == 0:
    print('\nAll fasting columns for manual inspection:')
    for c in sorted(fasting_matrix.columns):
        print(f'  {c}')

Fasting marker categories:
  inflammation: 1 markers
  glycemic: 0 markers
    (no matches — checking column names...)
  lipid_favorable: 18 markers
  lipid_adverse: 36 markers
  body_comp: 0 markers
    (no matches — checking column names...)

Postprandial marker categories:
  pp_glucose: 14 markers
  pp_lipid: 0 markers
  pp_inflammation: 5 markers


In [14]:
metabolic_scores = pd.DataFrame(index=range(K))

for ci in range(K):
    centroid = pd.Series(centroids_original[ci], index=important_species)
    
    for cat, markers in MARKER_CATEGORIES.items():
        if markers:
            vals = fasting_135[markers].T @ centroid
            metabolic_scores.loc[ci, f'fasting_{cat}'] = vals.mean()
    
    for cat, markers in pp_categories.items():
        if markers:
            vals = postprandial_135[markers].T @ centroid
            metabolic_scores.loc[ci, cat] = vals.mean()
    
    all_fasting = fasting_135.T @ centroid
    metabolic_scores.loc[ci, 'fasting_overall'] = all_fasting.mean()
    all_pp = postprandial_135.T @ centroid
    metabolic_scores.loc[ci, 'pp_overall'] = all_pp.mean()

metabolic_scores.index.name = 'cluster'

cols_to_plot = [c for c in metabolic_scores.columns if metabolic_scores[c].notna().all() and metabolic_scores[c].std() > 0]
if cols_to_plot:
    normed = metabolic_scores[cols_to_plot].apply(lambda x: (x - x.mean()) / x.std())
    fig = px.imshow(normed.values, x=cols_to_plot, y=[f'C{i}' for i in range(K)],
        color_continuous_scale='RdBu_r', aspect='auto',
        title='Metabolic Fingerprint per Cluster (z-scored)',
        template=PLOTLY_DARK, width=900, height=500)
    fig.show()
else:
    print('No valid metabolic category columns to plot — check marker name matching above')

print(metabolic_scores.round(6).to_string())

         fasting_inflammation  fasting_lipid_favorable  fasting_lipid_adverse  pp_glucose  pp_inflammation  fasting_overall  pp_overall
cluster                                                                                                                                
0                   -0.230314                -0.021819              -0.140644   -0.043348        -0.015504        -0.091531   -0.039502
1                   -0.380040                -0.038053              -0.230746   -0.030246        -0.045426        -0.153550   -0.048549
2                    0.149485                 0.025074               0.102155   -0.000735         0.021132         0.071521    0.016709
3                    0.121291                 0.008534               0.085169   -0.022506         0.026818         0.056399    0.007084
4                   -0.040993                 0.028340              -0.000896    0.006808        -0.001750         0.009998    0.001681
5                   -0.277155                -0.

---
## 8. Neuroactive Pathway Scores

In [15]:
neuro_scores = pd.DataFrame(index=range(K), columns=list(NEUROACTIVE_PATHWAYS.keys()))

for ci in range(K):
    centroid = pd.Series(centroids_original[ci], index=important_species)
    for pathway, species_list in NEUROACTIVE_PATHWAYS.items():
        if species_list:
            neuro_scores.loc[ci, pathway] = centroid[species_list].sum() / len(species_list)
        else:
            neuro_scores.loc[ci, pathway] = 0.0

neuro_scores = neuro_scores.astype(float)
neuro_scores.index.name = 'cluster'

neuro_normed = neuro_scores.apply(lambda x: (x - x.mean()) / x.std() if x.std() > 0 else 0)

fig = px.imshow(neuro_normed.values,
    x=list(NEUROACTIVE_PATHWAYS.keys()),
    y=[f'C{i}' for i in range(K)],
    color_continuous_scale='RdBu_r', aspect='auto',
    title='Neuroactive Pathway Scores per Cluster (z-scored)',
    template=PLOTLY_DARK, width=800, height=500)
fig.show()

print(neuro_scores.round(6).to_string())

         butyrate_SCFA      GABA  tryptophan_serotonin  dopamine_catecholamine  pro_inflammatory  gut_barrier
cluster                                                                                                      
0             0.007825  0.012614              0.033224               -0.044920         -0.036500     0.004111
1             0.050739 -0.006722              0.028812               -0.082677         -0.061872    -0.004450
2            -0.024678 -0.006939             -0.018756                0.016092          0.023982    -0.000105
3            -0.013775  0.007038             -0.004628                0.018074          0.024374     0.019303
4             0.007371 -0.021478             -0.002264               -0.012126         -0.002151    -0.006146
5             0.033875 -0.001874              0.024487               -0.062407         -0.046072     0.000483
6             0.020699 -0.007017              0.011486               -0.039205         -0.027800    -0.001010
7         

---
## 9. Dietary Pattern Alignment

In [16]:
pattern_df_raw = rows_to_df(load_strict_xlsx(TABLE5_FILE, 2))
pattern_matrix = pattern_df_raw.pivot(index='species', columns='variable', values='spearman_r')
pattern_135 = pattern_matrix.reindex(important_species).fillna(0)

key_indices = ['hPDI', 'uPDI', 'HFD', 'amed_score', 'hei_score']
available_indices = [k for k in key_indices if k in pattern_135.columns]
print(f'Available dietary indices: {available_indices}')

diet_alignment = pd.DataFrame(index=range(K))
for ci in range(K):
    centroid = pd.Series(centroids_original[ci], index=important_species)
    for idx in available_indices:
        diet_alignment.loc[ci, idx] = float(pattern_135[idx].values @ centroid.values)

diet_alignment.index.name = 'cluster'

if available_indices:
    da_normed = diet_alignment[available_indices].apply(lambda x: (x - x.mean()) / x.std() if x.std() > 0 else 0)
    fig = px.imshow(da_normed.values, x=available_indices, y=[f'C{i}' for i in range(K)],
        color_continuous_scale='RdBu_r', aspect='auto',
        title='Dietary Pattern Alignment per Cluster (z-scored)',
        template=PLOTLY_DARK, width=700, height=500)
    fig.show()

print(diet_alignment.round(6).to_string())

Available dietary indices: ['hPDI', 'uPDI', 'HFD', 'amed_score', 'hei_score']


             hPDI      uPDI       HFD  amed_score  hei_score
cluster                                                     
0        0.147209 -0.209884  0.069462    0.174060   0.138175
1        0.461425 -0.442562  0.277814    0.422355   0.413662
2       -0.312304  0.210123 -0.193077   -0.207151  -0.285894
3       -0.274729  0.232486 -0.156063   -0.178931  -0.249204
4       -0.090446 -0.014144 -0.084600   -0.013865  -0.070683
5        0.326722 -0.313978  0.192764    0.303034   0.286930
6        0.160960 -0.175920  0.089806    0.173311   0.141534
7        0.041744 -0.029289  0.031554    0.053357   0.036016
8        0.064245 -0.130020 -0.106011    0.102225   0.045050
9        0.205425 -0.200066 -0.048881    0.160996   0.141169
10       0.241691 -0.310367  0.099557    0.263100   0.228013
11      -0.259737  0.204109 -0.128303   -0.177649  -0.233641
12       0.037049 -0.085459 -0.006097    0.077575   0.029496
13      -0.273466  0.190547 -0.172117   -0.172023  -0.242959
14       0.577555 -0.544

---
## 10. Species-Level Highlights

In [17]:
cluster_species_data = []

for ci in range(K):
    centroid = pd.Series(centroids_original[ci], index=important_species)
    top5 = centroid.nlargest(5)
    bot5 = centroid.nsmallest(5)
    
    for sp, val in list(top5.items()) + list(bot5.items()):
        rank = t9.loc[sp, 'Final Rank'] if sp in t9.index else None
        food_corrs = corr_matrix.loc[sp] if sp in corr_matrix.index else pd.Series()
        top_food = food_corrs.abs().nlargest(1).index[0] if len(food_corrs) > 0 else ''
        top_food_r = food_corrs[top_food] if top_food else 0
        
        pathways = [p for p, slist in NEUROACTIVE_PATHWAYS.items() if sp in slist]
        
        cluster_species_data.append({
            'cluster': ci,
            'species': sp,
            'centroid_value': val,
            'direction': 'high' if val in top5.values else 'low',
            'health_rank': rank,
            'top_food_group': top_food,
            'top_food_r': top_food_r,
            'neuroactive_pathways': ', '.join(pathways) if pathways else 'none',
        })

cluster_top_species = pd.DataFrame(cluster_species_data)
print(f'Species highlights: {len(cluster_top_species)} entries across {K} clusters')

for ci in range(K):
    sub = cluster_top_species[cluster_top_species['cluster'] == ci]
    print(f'\n--- Cluster {ci} (health={centroid_health[ci]:.3f}) ---')
    for _, row in sub.iterrows():
        arrow = '↑' if row['direction'] == 'high' else '↓'
        rank_str = f'rank={row["health_rank"]:.0f}' if pd.notna(row['health_rank']) else 'no rank'
        print(f'  {arrow} {row["species"]}: val={row["centroid_value"]:.4f}, {rank_str}, food={row["top_food_group"]}({row["top_food_r"]:+.3f}), pathways={row["neuroactive_pathways"]}')

Species highlights: 160 entries across 16 clusters

--- Cluster 0 (health=1.616) ---
  ↑ Streptococcus_thermophilus: val=0.1670, rank=72, food=Dairy(+0.254), pathways=tryptophan_serotonin
  ↑ Bifidobacterium_animalis: val=0.1389, rank=29, food=Dairy(+0.199), pathways=GABA
  ↑ Firmicutes_bacterium_CAG_95: val=0.0732, rank=10, food=Nuts(+0.128), pathways=none
  ↑ Bacteroides_ovatus: val=0.0683, rank=92, food=Legumes(+0.135), pathways=none
  ↑ Fretibacterium_fastidiosum: val=0.0629, rank=48, food=Sweets_desserts(+0.088), pathways=none
  ↓ Anaerotruncus_colihominis: val=-0.0751, rank=165, food=Nuts(-0.142), pathways=none
  ↓ Clostridium_bolteae: val=-0.0684, rank=167, food=Nuts(-0.134), pathways=pro_inflammatory
  ↓ Flavonifractor_plautii: val=-0.0621, rank=168, food=Nuts(-0.120), pathways=pro_inflammatory
  ↓ Holdemania_filiformis: val=-0.0598, rank=129, food=Dairy(-0.081), pathways=none
  ↓ Clostridium_bolteae_CAG_59: val=-0.0589, rank=161, food=Egg(-0.101), pathways=pro_inflammatory

--

---
## 11. Export All Profiles

In [18]:
cluster_profiles = pd.DataFrame(index=range(K))
cluster_profiles.index.name = 'cluster'

cluster_profiles['member_count'] = [int((cluster_labels == ci).sum()) for ci in range(K)]
cluster_profiles['health_score'] = centroid_health
cluster_profiles['PC1'] = centroids_3d[:, 0]
cluster_profiles['PC2'] = centroids_3d[:, 1]
cluster_profiles['PC3'] = centroids_3d[:, 2]

for ci in range(K):
    top = pd.Series([base_labels[j] for j in range(len(base_labels)) if cluster_labels[j]==ci]).value_counts().head(3)
    cluster_profiles.loc[ci, 'top_diets'] = '; '.join([f'{d}({c})' for d,c in top.items()])

for col in metabolic_scores.columns:
    cluster_profiles[f'metabolic_{col}'] = metabolic_scores[col].values

for col in neuro_scores.columns:
    cluster_profiles[f'neuro_{col}'] = neuro_scores[col].values

for col in diet_alignment.columns:
    cluster_profiles[f'diet_{col}'] = diet_alignment[col].values

cluster_profiles.to_csv(OUT_DIR / 'cluster_profiles.csv')
cluster_top_species.to_csv(OUT_DIR / 'cluster_top_species.csv', index=False)

loadings.to_csv(OUT_DIR / 'pca_loadings.csv')

arch_df = pd.DataFrame(ALL_ARCHETYPES, index=FOOD_GROUPS).T
arch_df.index.name = 'archetype'
arch_df.to_csv(OUT_DIR / 'archetype_definitions.csv')

centroid_df = pd.DataFrame(centroids_original, columns=important_species)
centroid_df['health_score'] = centroid_health
centroid_df.index.name = 'cluster'
centroid_df.to_csv(OUT_DIR / 'cluster_centroids.csv')

fasting_matrix.to_csv(OUT_DIR / 'fasting_species_correlations.csv')
postprandial_matrix.to_csv(OUT_DIR / 'postprandial_species_correlations.csv')

print('Exported:')
for f in sorted(OUT_DIR.glob('*.csv')):
    print(f'  {f.name} ({f.stat().st_size:,} bytes)')

print(f'\nCluster profiles shape: {cluster_profiles.shape}')
print(cluster_profiles.to_string())

Exported:
  archetype_definitions.csv (6,282 bytes)
  cluster_centroids.csv (49,063 bytes)
  cluster_profiles.csv (8,554 bytes)
  cluster_top_species.csv (16,113 bytes)
  fasting_species_correlations.csv (261,655 bytes)
  food_category_weights.csv (8,532 bytes)
  food_species_profiles.csv (257,402 bytes)
  health_direction_vector.csv (5,857 bytes)
  important_species_list.csv (3,199 bytes)
  pca_loadings.csv (11,563 bytes)
  postprandial_species_correlations.csv (588,766 bytes)
  predict1_food_species_correlations.csv (73,065 bytes)
  predict1_food_species_qvalues.csv (67,672 bytes)
  predict1_significant_food_species.csv (31,748 bytes)
  species_importance.csv (27,865 bytes)

Cluster profiles shape: (16, 24)
         member_count  health_score        PC1        PC2        PC3                                                         top_diets  metabolic_fasting_inflammation  metabolic_fasting_lipid_favorable  metabolic_fasting_lipid_adverse  metabolic_pp_glucose  metabolic_pp_inflammati